In [ ]:
# 单元格0：UL-94 vocab 频率矩阵生成（真实表头适配 + 匹配优化 + 共聚/共混支持）
# Cell 0: canonical SMILES vocab feature generation (Cell 1 unchanged)
from motif_vocab_features import (FEATURE_FILTER_VERSION, MATCHING_VERSION, find_motif_root, run_feature_cell)
MOTIF_ROOT = find_motif_root()
VOCAB_FILE = MOTIF_ROOT / 'results' / 'local_vocab_parallel_threshold.csv'
if not VOCAB_FILE.exists():
    raise FileNotFoundError(f'Canonical vocab file not found: {VOCAB_FILE}')
MODEL_RESULTS_DIR = MOTIF_ROOT / 'results' / 'ul94_motif_select'
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
notebook_dir = MOTIF_ROOT / 'scripts' / 'feature_selection'
results_dir = str(MODEL_RESULTS_DIR)
FORCE_RECOMPUTE_FEATURES = True
feature_data = run_feature_cell('UL94', force_recompute=FORCE_RECOMPUTE_FEATURES)

'''Legacy Cell 0 implementation retained as inert reference; do not execute.
import re
import json
import time
import math
import hashlib
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

warnings.filterwarnings("ignore")
np.random.seed(42)
NUMERIC_ZERO_ATOL = 1e-6

# =============================================================================
# 基础配置
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("loi_motif_feature_matrix")


def resolve_motif_root(start: Optional[Path] = None) -> Path:
    """自动定位 LCMWR 根目录；支持从项目根、任意 scripts 子目录或 notebook 启动。"""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, start / "LCMWR", *start.parents]
    for path in candidates:
        if path.name.lower() == "lcmwr" and all((path / part).exists() for part in ("dataset", "results", "scripts")):
            return path
    raise FileNotFoundError(f"无法定位 LCMWR 根目录：{start}")


MOTIF_ROOT = resolve_motif_root()
DATA_DIR = MOTIF_ROOT / "data"
DATASET_DIR = MOTIF_ROOT / "dataset"
RESULTS_DIR = MOTIF_ROOT / "results"
MODEL_RESULTS_DIR = RESULTS_DIR / "ul94_motif_select"
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 默认优先读取当前性能数据集单独生成的 vocab，避免把全数据集 vocab 数误用为原始特征数。
TASK_VOCAB_CANDIDATES = [
    MODEL_RESULTS_DIR / "ul94_local_vocab_parallel_threshold.csv",
    RESULTS_DIR / "ul94_local_vocab_parallel_threshold.csv",
    RESULTS_DIR / "local_vocab_parallel_threshold_ul94.csv",
]
GLOBAL_VOCAB_FILE = RESULTS_DIR / "local_vocab_parallel_threshold.csv"
LEGACY_VOCAB_FILE = RESULTS_DIR / "local_fragments_parallel_threshold.csv"
VOCAB_FILE = next((p for p in TASK_VOCAB_CANDIDATES if p.exists()), GLOBAL_VOCAB_FILE)
if not VOCAB_FILE.exists() and LEGACY_VOCAB_FILE.exists():
    VOCAB_FILE = LEGACY_VOCAB_FILE
UL94_DATA_FILE = DATASET_DIR / "UL-94.csv"

# 与 Cell 1 兼容：Cell 1 会继续使用 results_dir 变量
notebook_dir = MOTIF_ROOT / "scripts" / "feature_selection"
results_dir = str(MODEL_RESULTS_DIR)
os.makedirs(results_dir, exist_ok=True)

logger.info("UL-94 vocab weighted feature module loaded")
logger.info("MOTIF_ROOT: %s", MOTIF_ROOT)
logger.info("Default vocab file: %s", VOCAB_FILE)
logger.info("Default UL-94 file: %s", UL94_DATA_FILE)

# =============================================================================
# 工具函数：路径、hash、列名与 SMILES 预处理
# =============================================================================


def file_sha1(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()[:12]


def preprocess_smiles(smiles: Any) -> Any:
    """沿用原逻辑：将聚合物连接占位符 [Fr]/[Rb] 替换为 [H]。"""
    if not isinstance(smiles, str):
        return smiles
    smiles = smiles.strip()
    if not smiles:
        return np.nan
    return smiles.replace("[Fr]", "[H]").replace("[Rb]", "[H]")


def calculate_molecular_weight(smiles: Any) -> float:
    smiles = preprocess_smiles(smiles)
    if not isinstance(smiles, str):
        return np.nan
    mol = Chem.MolFromSmiles(smiles)
    return Chem.rdMolDescriptors.CalcExactMolWt(mol) if mol is not None else np.nan


def safe_feature_name(name: str) -> str:
    """生成与旧脚本基本兼容的 vocab 特征列名。"""
    s = str(name)
    s = s.replace("[", "LB_").replace("]", "_RB")
    s = s.replace("<", "LT_").replace(">", "_GT")
    s = s.replace("/", "_slash_").replace("\\", "_backslash_")
    s = re.sub(r"\s+", "_", s)
    return s


def deduplicate_names(names: List[str]) -> List[str]:
    counts: Dict[str, int] = {}
    out = []
    for n in names:
        if n not in counts:
            counts[n] = 0
            out.append(n)
        else:
            counts[n] += 1
            out.append(f"{n}__dup{counts[n]}")
    return out


def read_csv_auto(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="iso-8859-1")


def detect_vocab_column(vocab_df: pd.DataFrame) -> str:
    candidates = ["vocab", "Vocab", "smiles", "vocab_smiles", "fragment", "fragment_smiles", "canonical_smiles"]
    for col in candidates:
        if col in vocab_df.columns:
            return col
    raise ValueError(f"vocab 表缺少可识别的 SMILES/SMARTS 列，候选列名包括：{candidates}")


def present(row: pd.Series, col: str) -> bool:
    if col not in row.index:
        return False
    val = row[col]
    if pd.isna(val):
        return False
    if isinstance(val, str) and not val.strip():
        return False
    return True


def get_numeric(row: pd.Series, col: str) -> float:
    if col not in row.index:
        return np.nan
    try:
        val = float(row[col])
    except Exception:
        return np.nan
    return val if np.isfinite(val) else np.nan


def normalize_amounts(amounts: List[float]) -> List[float]:
    total = float(np.nansum(amounts))
    if total <= 0 or not np.isfinite(total):
        return [np.nan for _ in amounts]
    return [float(a) / total for a in amounts]

# =============================================================================
# vocab 匹配优化：预编译 query mol + unique SMILES 缓存
# =============================================================================


def compile_single_vocab(vocab: Any) -> Optional[Chem.Mol]:
    """优先按 SMARTS 编译，失败后按 SMILES 编译。"""
    if not isinstance(vocab, str) or not vocab.strip():
        return None
    frag = preprocess_smiles(vocab)
    q = None
    try:
        q = Chem.MolFromSmarts(frag)
    except Exception:
        q = None
    if q is not None:
        return q
    try:
        q = Chem.MolFromSmiles(frag)
    except Exception:
        q = None
    return q


def prepare_vocab(vocab_file_path: Path) -> Tuple[pd.DataFrame, List[str], List[str], List[Chem.Mol]]:
    vocab_df = read_csv_auto(vocab_file_path)

    # 如果使用的是“达标筛选”基元表，默认只取已通过筛选的片段。
    if "passed_support_filter" in vocab_df.columns:
        before = len(vocab_df)
        mask = vocab_df["passed_support_filter"].astype(str).str.lower().isin(["true", "1", "yes"])
        if mask.any():
            vocab_df = vocab_df.loc[mask].copy()
            logger.info("Using passed_support_filter=True fragments: %d -> %d", before, len(vocab_df))

    vocab_col = detect_vocab_column(vocab_df)
    raw_vocab = [str(x).strip() for x in vocab_df[vocab_col].tolist() if isinstance(x, str) and str(x).strip()]

    compiled: List[Chem.Mol] = []
    valid_vocab: List[str] = []
    invalid_vocab: List[str] = []
    seen = set()
    for frag in raw_vocab:
        frag = preprocess_smiles(frag)
        if frag in seen:
            continue
        seen.add(frag)
        q = compile_single_vocab(frag)
        if q is None:
            invalid_vocab.append(frag)
            continue
        valid_vocab.append(frag)
        compiled.append(q)

    feature_names = deduplicate_names([safe_feature_name(f) for f in valid_vocab])
    logger.info("Vocabulary loaded: raw=%d, valid=%d, invalid=%d", len(raw_vocab), len(valid_vocab), len(invalid_vocab))
    if invalid_vocab:
        logger.warning("Invalid vocab entries skipped, first examples: %s", invalid_vocab[:5])

    return vocab_df, valid_vocab, feature_names, compiled


def count_vocab_for_smiles(smiles: Any, query_mols: List[Chem.Mol]) -> np.ndarray:
    """对单个 SMILES 统计所有 vocab 出现次数。query_mols 已预编译，避免重复 MolFromSmarts。"""
    arr = np.zeros(len(query_mols), dtype=np.float32)
    smiles = preprocess_smiles(smiles)
    if not isinstance(smiles, str):
        return arr
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return arr
    for j, q in enumerate(query_mols):
        try:
            arr[j] = len(mol.GetSubstructMatches(q, uniquify=True))
        except Exception:
            arr[j] = 0.0
    return arr


def build_unique_smiles_feature_lookup(
    smiles_values: List[Any],
    query_mols: List[Chem.Mol],
    n_jobs: int = -1,
) -> Dict[str, np.ndarray]:
    cleaned = []
    for smi in smiles_values:
        smi = preprocess_smiles(smi)
        if isinstance(smi, str) and smi:
            cleaned.append(smi)
    unique_smiles = sorted(set(cleaned))
    logger.info("Unique component SMILES to featurize: %d", len(unique_smiles))

    start = time.time()
    vectors = Parallel(n_jobs=n_jobs, prefer="threads")(
        delayed(count_vocab_for_smiles)(smi, query_mols)
        for smi in unique_smiles
    )
    elapsed = (time.time() - start) / 60
    logger.info("Unique SMILES featurization completed in %.2f min", elapsed)

    return dict(zip(unique_smiles, vectors))

# =============================================================================
# 真实表头适配：共聚组分与共混组分解析
# =============================================================================

CO_SMILES_COLS = ["smiles1", "smiles2"]
CO_CONTENT_COLS = ["co_content1", "co_content2"]
CO_WT_COLS = ["wt1", "wt2"]
CO_MOL_COLS = ["mol1", "mol2"]

MIX_SLOTS = [1, 2, 3, 4]


def detect_table_schema(df: pd.DataFrame) -> Dict[str, Any]:
    required = ["smiles1", "smiles2", "UL-94"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"UL-94 数据表缺少必要列：{missing}")
    mix_slots = [i for i in MIX_SLOTS if f"mix_smiles{i}" in df.columns or f"mix_content{i}" in df.columns]
    return {
        "co_smiles_cols": [c for c in CO_SMILES_COLS if c in df.columns],
        "mix_slots": mix_slots,
    }


def co_component_amount(row: pd.Series, idx: int) -> float:
    """
    共聚组分用量。优先级：co_content_i > mol_i > wt_i/MW。
    注意：co_content 可以是 0-1、0-100 或任意配比，后续都会归一化为 x_i。
    """
    smi_col = f"smiles{idx}"
    smi = row.get(smi_col, np.nan)
    if not isinstance(smi, str) or not smi.strip():
        return np.nan

    content = get_numeric(row, f"co_content{idx}")
    if np.isfinite(content) and content > 0:
        return content

    mol_val = get_numeric(row, f"mol{idx}")
    if np.isfinite(mol_val) and mol_val > 0:
        return mol_val

    wt_val = get_numeric(row, f"wt{idx}")
    if np.isfinite(wt_val) and wt_val > 0:
        mw = calculate_molecular_weight(smi)
        if np.isfinite(mw) and mw > 0:
            return wt_val / mw

    return np.nan


def mix_component_amount(row: pd.Series, idx: int, smiles: Any) -> Tuple[float, str]:
    """
    共混组分用量统一换算为 mol 比例基准。

    优先级：mix_mol_i > mix_content_i > mix_wt_i/MW。
    mix_content_i 没有额外单位标记时按 mol 配比处理；mix_wt_i 通过对应 mix_smiles_i
    的分子量换算为 mol 数。后续会和固定为 100 的共聚主体一起归一化。
    """
    mol_val = get_numeric(row, f"mix_mol{idx}")
    if np.isfinite(mol_val) and mol_val > 0:
        return float(mol_val), "mix_mol"

    content = get_numeric(row, f"mix_content{idx}")
    if np.isfinite(content) and content > 0:
        return float(content), "mix_content_as_mol_ratio"

    wt_val = get_numeric(row, f"mix_wt{idx}")
    if np.isfinite(wt_val) and wt_val > 0:
        mw = calculate_molecular_weight(smiles)
        if np.isfinite(mw) and mw > 0:
            return float(wt_val) / mw, "mix_wt_to_mol"

    return np.nan, "missing"


def collect_copolymer_components(row: pd.Series, feature_lookup: Dict[str, np.ndarray]) -> List[Dict[str, Any]]:
    comps = []
    for idx in (1, 2):
        smi = preprocess_smiles(row.get(f"smiles{idx}", np.nan))
        if not isinstance(smi, str) or not smi:
            continue
        vec = feature_lookup.get(smi)
        if vec is None:
            continue
        amount = co_component_amount(row, idx)
        comps.append({"idx": idx, "smiles": smi, "amount": amount, "features": vec})

    # 如果只有 SMILES 但没有任何含量信息，默认等比例，避免整行被误删。
    finite_amounts = [c["amount"] for c in comps if np.isfinite(c["amount"]) and c["amount"] > 0]
    if comps and not finite_amounts:
        for c in comps:
            c["amount"] = 1.0
    return comps


def collect_mix_components(row: pd.Series, feature_lookup: Dict[str, np.ndarray]) -> List[Dict[str, Any]]:
    comps = []
    for idx in MIX_SLOTS:
        smi_col = f"mix_smiles{idx}"
        if smi_col not in row.index:
            continue
        smi = preprocess_smiles(row.get(smi_col, np.nan))
        if not isinstance(smi, str) or not smi:
            continue
        vec = feature_lookup.get(smi)
        if vec is None:
            continue
        amount, amount_source = mix_component_amount(row, idx, smi)
        if not np.isfinite(amount) or amount <= 0:
            # 有 mix_smiles 但没有可用 mol/weight/content 用量时跳过该 mix 组分。
            continue
        comps.append({"idx": idx, "smiles": smi, "amount": float(amount), "amount_source": amount_source, "features": vec})
    return comps


def has_mix_information(row: pd.Series) -> bool:
    """只要存在 mix_smiles 或 mix_content/mix_wt/mix_mol，就认为该行包含共混信息。"""
    for idx in MIX_SLOTS:
        for prefix in ("mix_smiles", "mix_content", "mix_wt", "mix_mol"):
            col = f"{prefix}{idx}"
            if present(row, col):
                return True
    return False


def weighted_linear_feature(components: List[Dict[str, Any]], n_features: int, prefix: str = "co") -> Tuple[np.ndarray, Dict[str, Any]]:
    """共聚物规则：F_co = Σ x_i * F_i。"""
    if not components:
        return np.zeros(n_features, dtype=np.float32), {f"{prefix}_mode": "invalid", f"{prefix}_n_components": 0}

    valid_components = [c for c in components if np.isfinite(c["amount"]) and c["amount"] > 0]
    if not valid_components:
        return np.zeros(n_features, dtype=np.float32), {f"{prefix}_mode": "invalid", f"{prefix}_n_components": len(components)}

    total = sum(c["amount"] for c in valid_components)
    out = np.zeros(n_features, dtype=np.float32)
    weights = {}
    for c in valid_components:
        w = c["amount"] / total
        out += w * c["features"]
        weights[f"{prefix}_x{c['idx']}"] = w
    return out, {f"{prefix}_mode": "copolymer", f"{prefix}_n_components": len(valid_components), **weights}


def inverse_blend_feature(components: List[Tuple[float, np.ndarray, str]], n_features: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    共混物规则：1 / (1 + F_blend) = Σ x_i / (1 + F_i)。
    即 F_blend = 1 / Σ[x_i / (1 + F_i)] - 1。
    """
    valid_components = [(a, v, n) for a, v, n in components if np.isfinite(a) and a > 0]
    if not valid_components:
        return np.zeros(n_features, dtype=np.float32), {"blend_mode": "invalid", "blend_n_components": 0}
    total = sum(amount for amount, _, _ in valid_components)
    if total <= 0:
        return np.zeros(n_features, dtype=np.float32), {"blend_mode": "invalid", "blend_n_components": len(valid_components)}

    component_vectors = [np.asarray(vec, dtype=np.float64) for _, vec, _ in valid_components]
    active_feature_mask = np.zeros(n_features, dtype=bool)
    for vec in component_vectors:
        active_feature_mask |= np.abs(vec) > NUMERIC_ZERO_ATOL

    out = np.zeros(n_features, dtype=np.float64)
    weights = {}
    eps = 1e-12
    for amount, _, name in valid_components:
        weights[f"blend_x_{name}"] = amount / total

    # 如果所有组分在某个 vocab 上都是 0，该 vocab 的 blend 输出应严格为 0；
    # 只在 active 位置计算 inverse rule，避免 Σx_i 的 float32 舍入误差产生 1.19e-07 假非零。
    if active_feature_mask.any():
        denom = np.zeros(int(active_feature_mask.sum()), dtype=np.float64)
        for (amount, _, _), vec in zip(valid_components, component_vectors):
            w = amount / total
            denom += w / np.maximum(1.0 + vec[active_feature_mask], eps)
        active_out = 1.0 / np.maximum(denom, eps) - 1.0
        active_out[np.abs(active_out) <= NUMERIC_ZERO_ATOL] = 0.0
        active_out[active_out < 0] = np.where(
            active_out[active_out < 0] > -NUMERIC_ZERO_ATOL,
            0.0,
            active_out[active_out < 0],
        )
        out[active_feature_mask] = active_out

    return out.astype(np.float32), {"blend_mode": "blend", "blend_n_components": len(valid_components), **weights}


def combine_row_features(
    row: pd.Series,
    feature_lookup: Dict[str, np.ndarray],
    n_features: int,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    真实表头下的组合规则：
    1) 共聚部分：smiles1/smiles2 按 co_content1/2、mol1/2 或 wt1/2 计算 F_co = Σ x_i F_i。
    2) 若无 mix 信息：直接返回 F_co。
    3) 若有 mix 信息：先把 F_co 作为用量 100 的共混主体；mix 组分按 mol 比例加入，
       mix_wt_i 先用分子量换算为 mol。所有共混组分再归一化为 x_i，并按
       1/(1+F_blend)=Σ[x_i/(1+F_i)] 计算。
    """
    co_components = collect_copolymer_components(row, feature_lookup)
    co_vec, co_meta = weighted_linear_feature(co_components, n_features, prefix="co")

    if co_meta.get("co_mode") != "copolymer":
        return np.zeros(n_features, dtype=np.float32), {"composition_mode": "invalid_copolymer", **co_meta}

    mix_present = has_mix_information(row)
    if not mix_present:
        return co_vec, {"composition_mode": "copolymer", **co_meta, "mix_detected": False}

    mix_components = collect_mix_components(row, feature_lookup)

    # 共混物中，前面的共聚主体先计算 F_co，然后固定作为用量 100 的组分。
    # 后续 mix 组分用 mol 基准量加入；inverse_blend_feature 内部会统一归一化。
    co_blend_amount = 100.0
    blend_components: List[Tuple[float, np.ndarray, str]] = [(co_blend_amount, co_vec, "copolymer")]
    mix_amount_sources = {}
    for c in mix_components:
        blend_components.append((c["amount"], c["features"], f"mix{c['idx']}"))
        mix_amount_sources[f"mix_amount_source{c['idx']}"] = c.get("amount_source", "unknown")

    blend_vec, blend_meta = inverse_blend_feature(blend_components, n_features)
    meta = {
        "composition_mode": "blend",
        **co_meta,
        "mix_detected": True,
        "mix_n_components": len(mix_components),
        "copolymer_blend_amount": co_blend_amount,
        "copolymer_blend_amount_source": "fixed_100",
        **mix_amount_sources,
        **blend_meta,
    }
    return blend_vec, meta

# =============================================================================
# 主函数
# =============================================================================


def generate_feature_matrix(
    vocab_file_path: Any = VOCAB_FILE,
    data_file_path: Any = UL94_DATA_FILE,
    n_jobs: int = -1,
    use_cache: bool = True,
    force_recompute: bool = False,
) -> Dict[str, Any]:
    """生成 UL-94 样本级原始 vocab 加权特征表，支持当前真实表头下的共聚物和共混物组合。"""
    vocab_file_path = Path(vocab_file_path)
    data_file_path = Path(data_file_path)

    logger.info("Starting UL-94 feature matrix generation")
    logger.info("Vocab file: %s", vocab_file_path)
    logger.info("Data file: %s", data_file_path)

    if not vocab_file_path.exists():
        raise FileNotFoundError(f"Vocabulary file not found: {vocab_file_path}")
    if not data_file_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_file_path}")

    vocab_df, valid_vocab, feature_names, query_mols = prepare_vocab(vocab_file_path)
    smi_df = read_csv_auto(data_file_path)
    logger.info("Data file loaded: %d samples", len(smi_df))

    schema = detect_table_schema(smi_df)
    if "UL-94" not in smi_df.columns:
        raise ValueError("UL-94 数据表缺少 UL-94 列")

    DF = smi_df.copy()
    smiles_cols_to_clean = ["co_smiles", "co_smiles1", "smiles1", "smiles2"]
    smiles_cols_to_clean += [f"mix_smiles{i}" for i in MIX_SLOTS]
    for col in smiles_cols_to_clean:
        if col in DF.columns:
            DF[col] = DF[col].apply(preprocess_smiles)

    initial_count = len(DF)
    DF = DF.dropna(subset=["UL-94"]).copy()
    logger.info("After cleaning UL-94: %d samples (removed %d)", len(DF), initial_count - len(DF))

    vocab_hash = file_sha1(vocab_file_path)
    data_hash = file_sha1(data_file_path)
    cache_key = "ul94_vocab_weighted_v5"
    feature_filter_version = "drop_near_zero_columns_v3_blend_active_mask"
    zero_feature_atol = NUMERIC_ZERO_ATOL
    feature_cache = MODEL_RESULTS_DIR / f"{cache_key}.pkl"
    processed_data_file = MODEL_RESULTS_DIR / "ul94_vocab_processed_data.csv"
    feature_matrix_file = MODEL_RESULTS_DIR / "ul94_vocab_weighted_feature_matrix.csv"
    weighted_table_file = MODEL_RESULTS_DIR / "ul94_vocab_weighted_raw_features.csv"
    stats_file = MODEL_RESULTS_DIR / "ul94_vocab_feature_matrix_stats.json"

    if use_cache and (not force_recompute) and feature_cache.exists() and weighted_table_file.exists():
        try:
            cached = joblib.load(feature_cache)
            if (
                cached.get("vocab_hash") == vocab_hash
                and cached.get("data_hash") == data_hash
                and cached.get("blend_rule_version") == "fixed100_molblend_v1"
                and cached.get("feature_filter_version") == feature_filter_version
            ):
                logger.info("Loading UL-94 vocab weighted features from cache: %s", feature_cache)
                cached["weighted_raw_feature_table_file"] = str(weighted_table_file)
                return cached
            logger.info("Ignoring stale UL-94 feature cache because vocab/data hash changed: %s", feature_cache)
        except Exception as exc:
            logger.warning("Existing UL-94 feature cache could not be reused: %s", exc)

    if use_cache and (not force_recompute) and weighted_table_file.exists() and feature_matrix_file.exists() and processed_data_file.exists() and stats_file.exists():
        try:
            stats_existing = json.loads(stats_file.read_text(encoding="utf-8"))
            if (
                stats_existing.get("vocab_hash") == vocab_hash
                and stats_existing.get("data_hash") == data_hash
                and stats_existing.get("blend_rule_version") == "fixed100_molblend_v1"
                and stats_existing.get("feature_filter_version") == feature_filter_version
            ):
                logger.info("Loading existing UL-94 vocab weighted CSV outputs without recomputing: %s", weighted_table_file)
                DF_cached = read_csv_auto(processed_data_file)
                X_cached = read_csv_auto(feature_matrix_file)
                cached_feature_names = list(X_cached.columns)
                cached_feature_name_set = set(cached_feature_names)
                cached_vocab = [
                    frag for frag, name in zip(valid_vocab, feature_names)
                    if name in cached_feature_name_set
                ]
                y_cached = DF_cached["UL-94"].astype(str)
                return {
                    "processed_data": DF_cached,
                    "feature_matrix": X_cached,
                    "target_variable": y_cached,
                    "vocab_smiles": cached_vocab,
                    "vocab_fragments": cached_vocab,
                    "feature_names": cached_feature_names,
                    "vocab_file": str(vocab_file_path),
                    "data_file": str(data_file_path),
                    "weighted_raw_feature_table_file": str(weighted_table_file),
                    "blend_rule_version": "fixed100_molblend_v1",
                    "feature_filter_version": feature_filter_version,
                    "loaded_from_existing_outputs": True,
                    "schema": stats_existing.get("schema", {}),
                }
        except Exception as exc:
            logger.warning("Existing UL-94 vocab outputs could not be reused: %s", exc)

    # 只对真正参与基元匹配的组分 SMILES 建特征：smiles1/2 和 mix_smiles1-4。
    all_component_smiles = []
    for col in ["smiles1", "smiles2"] + [f"mix_smiles{i}" for i in MIX_SLOTS]:
        if col in DF.columns:
            all_component_smiles.extend(DF[col].tolist())
    feature_lookup = build_unique_smiles_feature_lookup(all_component_smiles, query_mols, n_jobs=n_jobs)

    logger.info("Combining component features for %d UL-94 samples", len(DF))
    start = time.time()
    combined = []
    meta_rows = []
    n_features = len(valid_vocab)
    for _, row in DF.iterrows():
        vec, meta = combine_row_features(row, feature_lookup, n_features)
        combined.append(vec)
        meta_rows.append(meta)
    elapsed = (time.time() - start) / 60
    logger.info("Feature combination completed in %.2f min", elapsed)

    X_combined = pd.DataFrame(np.vstack(combined), columns=feature_names, index=DF.index)
    X_combined = X_combined.replace([np.inf, -np.inf], np.nan).fillna(0)

    feature_count_before_zero_filter = int(X_combined.shape[1])
    near_zero_mask = X_combined.abs() <= zero_feature_atol
    if bool(near_zero_mask.to_numpy().any()):
        X_combined = X_combined.mask(near_zero_mask, 0.0)
    nonzero_feature_mask = (X_combined.abs() > zero_feature_atol).any(axis=0).to_numpy(dtype=bool)
    zero_feature_count_removed = int((~nonzero_feature_mask).sum())
    if zero_feature_count_removed:
        logger.info(
            "Dropping near-zero vocab features with abs(value) <= %.1e: %d -> %d",
            zero_feature_atol,
            feature_count_before_zero_filter,
            int(nonzero_feature_mask.sum()),
        )
        X_combined = X_combined.loc[:, nonzero_feature_mask].copy()
        valid_vocab = [
            frag for frag, keep in zip(valid_vocab, nonzero_feature_mask)
            if bool(keep)
        ]
        feature_names = list(X_combined.columns)

    meta_df = pd.DataFrame(meta_rows, index=DF.index)
    DF = pd.concat([DF, meta_df], axis=1)

    valid_mask = DF["composition_mode"].isin(["copolymer", "blend"])
    if not valid_mask.all():
        removed = int((~valid_mask).sum())
        logger.warning("Removing invalid composition samples: %d", removed)
        logger.warning("Invalid composition modes: %s", DF.loc[~valid_mask, "composition_mode"].value_counts().to_dict())
        DF = DF.loc[valid_mask].copy()
        X_combined = X_combined.loc[valid_mask].copy()

    y = DF["UL-94"].astype(str)

    logger.info("Feature matrix info: samples=%d, features=%d", X_combined.shape[0], X_combined.shape[1])
    logger.info("Composition modes: %s", DF["composition_mode"].value_counts().to_dict())

    weighted_raw_table = pd.concat([DF.reset_index(drop=True), X_combined.reset_index(drop=True)], axis=1)
    DF.to_csv(processed_data_file, index=False)
    X_combined.to_csv(feature_matrix_file, index=False)
    weighted_raw_table.to_csv(weighted_table_file, index=False)

    output = {
        "processed_data": DF,
        "feature_matrix": X_combined,
        "target_variable": y,
        "vocab_smiles": valid_vocab,
        "vocab_fragments": valid_vocab,
        "feature_names": feature_names,
        "vocab_file": str(vocab_file_path),
        "data_file": str(data_file_path),
        "composition_rule": {
            "copolymer": "F_co = sum(x_i * F_i), x_i from co_content_i > mol_i > wt_i/MW",
            "blend": "Copolymer part is first converted to F_co and used as amount 100; mix components use mol amounts, with mix_wt_i converted to mol by molecular weight. All blend amounts are normalized before 1/(1+F_blend)=sum(x_i/(1+F_i)).",
            "blend_copolymer_amount": "fixed_100",
        },
        "schema": schema,
        "vocab_hash": vocab_hash,
        "data_hash": data_hash,
        "blend_rule_version": "fixed100_molblend_v1",
        "feature_filter_version": feature_filter_version,
        "feature_count_before_zero_filter": feature_count_before_zero_filter,
        "zero_feature_count_removed": zero_feature_count_removed,
        "zero_feature_atol": zero_feature_atol,
        "weighted_raw_feature_table_file": str(weighted_table_file),
    }

    if use_cache:
        joblib.dump(output, feature_cache)
        logger.info("Feature data saved to cache: %s", feature_cache)

    stats = {
        "vocab_file": str(vocab_file_path),
        "data_file": str(data_file_path),
        "valid_vocab_count": len(valid_vocab),
        "sample_count": int(X_combined.shape[0]),
        "feature_count": int(X_combined.shape[1]),
        "feature_count_before_zero_filter": feature_count_before_zero_filter,
        "zero_feature_count_removed": zero_feature_count_removed,
        "zero_feature_atol": zero_feature_atol,
        "composition_modes": DF["composition_mode"].value_counts().to_dict(),
        "schema": schema,
        "vocab_hash": vocab_hash,
        "data_hash": data_hash,
        "blend_rule_version": "fixed100_molblend_v1",
        "feature_filter_version": feature_filter_version,
        "cache_file": str(feature_cache),
        "processed_data_file": str(processed_data_file),
        "feature_matrix_file": str(feature_matrix_file),
        "weighted_raw_feature_table_file": str(weighted_table_file),
    }
    with open(stats_file, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    logger.info("UL-94 vocab weighted feature generation completed")
    return output

# =============================================================================
# 运行特征矩阵生成
# =============================================================================

if VOCAB_FILE.exists() and UL94_DATA_FILE.exists():
    feature_data = generate_feature_matrix(
        vocab_file_path=VOCAB_FILE,
        data_file_path=UL94_DATA_FILE,
        n_jobs=-1,
        use_cache=True,
        force_recompute=False,
    )
    logger.info("Cell 0 completed. UL-94 vocab weighted raw feature table is ready; you can now run Cell 1 for feature selection analysis.")
else:
    feature_data = None
    logger.warning("默认输入文件不存在，已跳过自动运行。请检查 VOCAB_FILE 和 UL94_DATA_FILE 后手动调用 generate_feature_matrix().")
'''


In [ ]:
# =============================================================================
# 改进的750组合四维阈值敏感性分析 - 修复算法缺陷
# =============================================================================
# 设置
import os
import warnings
import numpy as np
from pathlib import Path
warnings.filterwarnings('ignore')
np.random.seed(42)

# 创建结果目录 - 这部分缺失了！
if 'MOTIF_ROOT' in globals() and 'MODEL_RESULTS_DIR' in globals():
    notebook_dir = MOTIF_ROOT / "scripts" / "feature_selection"
    results_dir = str(MODEL_RESULTS_DIR)
    plots_data_dir = str(MODEL_RESULTS_DIR / "plots_data")
else:
    notebook_dir = Path.cwd()
    if not (notebook_dir / 'UL94_motif_select.ipynb').exists():
        candidate_dir = Path('局域化学基元的生成与筛选结果/UL-94_motif')
        if candidate_dir.exists():
            notebook_dir = candidate_dir
    results_dir = str(notebook_dir / "model_results")
    plots_data_dir = str(notebook_dir / "plots_data")
os.makedirs(results_dir, exist_ok=True)
os.makedirs(plots_data_dir, exist_ok=True)
mpl_config_dir = notebook_dir / ".matplotlib_cache"
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config_dir))
os.makedirs(mpl_config_dir, exist_ok=True)

import matplotlib
if not hasattr(matplotlib.rcParams, "_get"):
    matplotlib.rcParams._get = matplotlib.rcParams.get
matplotlib.use("Agg", force=True)

print("Improved 750-Combination 4D Threshold Sensitivity Analysis")
print("UL94 selection metric: ROC-AUC")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns  # optional; this notebook does not require it for selection
except ImportError:
    sns = None
from sklearn.model_selection import cross_val_score, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, make_scorer, roc_auc_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
import xgboost as xgb
import time
import json
from itertools import product
from scipy.stats import randint, uniform, pearsonr
from functools import partial

if 'MOTIF_ROOT' in globals() and 'MODEL_RESULTS_DIR' in globals():
    notebook_dir = MOTIF_ROOT / "scripts" / "feature_selection"
    results_dir = str(MODEL_RESULTS_DIR)
    plots_data_dir = str(MODEL_RESULTS_DIR / "plots_data")
else:
    notebook_dir = Path.cwd()
    if not (notebook_dir / 'UL94_motif_select.ipynb').exists():
        candidate_dir = Path('局域化学基元的生成与筛选结果/UL-94_motif')
        if candidate_dir.exists():
            notebook_dir = candidate_dir
    results_dir = str(notebook_dir / "model_results")
    plots_data_dir = str(notebook_dir / "plots_data")

# =============================================================================
# 审稿复现模式配置
# =============================================================================
# 说明：
# 1) 默认优先复用论文阶段已经保存的最终特征矩阵，避免随机因素/特征顺序导致最终特征数漂移；
# 2) 750 组合搜索结果默认优先读取已有缓存，除非显式设置 RECOMPUTE_750_SEARCH=True；
# 3) 互信息、模型评估和特征合并顺序均做确定性处理。
RANDOM_STATE = 48  # Fixed seed shared with the completed LOI nested 5×3 search.
OUTER_CV_SPLITS = 5
INNER_CV_SPLITS = 3
N_ITER_SEARCH = 20
np.random.seed(RANDOM_STATE)

REPRODUCE_PAPER_FEATURES = False     # AUC 筛选后重新生成最终特征集
SELECTION_SCORING = 'roc_auc'       # UL94 从 750 搜索起统一以 ROC-AUC 评分
SELECTION_METRIC_LABEL = 'ROC-AUC'
USE_750_CACHE = False                # AUC 与原 ROC-AUC 协议不同，禁止复用旧缓存
RECOMPUTE_750_SEARCH = True          # True：进入搜索函数；若 checkpoint 存在则从半截继续
RECOMPUTE_FINAL_FEATURES = True      # True：750 续跑完成后重新筛选生成最终矩阵
# 两层并行总并发约为 2 × 4，适配 8 个逻辑 CPU。
XGB_N_JOBS = 4
CV_SEARCH_N_JOBS = 2

MI_FUNC = partial(mutual_info_classif, random_state=RANDOM_STATE)

UL94_LABEL_NAMES = {0: "non-V-0", 1: "V-0"}


def normalize_ul94_target(y_raw):
    """读取已标准化的 UL-94 标签：V-0=1，non-V-0=0。"""
    labels = pd.Series(y_raw).astype(str).str.strip()
    labels = labels.str.replace("－", "-", regex=False).str.replace("–", "-", regex=False)
    return labels.map({"V-0": 1, "non-V-0": 0})


def ordered_union(selected, protected):
    """保序合并特征列表，避免 set() 打乱特征顺序影响相关性筛选结果。"""
    return list(dict.fromkeys(list(selected) + list(protected)))

from tqdm import tqdm
warnings.filterwarnings('ignore')

print("Improved 750-Combination 4D Threshold Sensitivity Analysis")
print("=" * 70)
print("Key improvements:")
print("✓ Removed random sampling in correlation filtering")
print("✓ Fixed early stopping mechanism") 
print("✓ Halogen vocab protection only in frequency/variance filtering")
print("✓ Improved exception handling")
print("✓ Enhanced algorithm robustness")
print("=" * 70)

# 检查数据源
if 'feature_data' in globals():
    # 直接从Cell 0的特征矩阵生成结果获取数据
    original_X = feature_data['feature_matrix']
    y = feature_data['target_variable']
    
    # 识别 Br/F/Cl 阻燃相关元素 vocab 特征；只在频率和方差筛选阶段保护。
    print("Identifying Br / F / Cl halogen vocab features from feature matrix...")
    
    halogen_feature_names = {
        'Br', 'F', 'Cl',
        'LB_Br_RB', 'LB_F_RB', 'LB_Cl_RB',
        'LB_Br-_RB', 'LB_F-_RB', 'LB_Cl-_RB',
        'LB_Br+_RB', 'LB_F+_RB', 'LB_Cl+_RB',
    }
    potential_whitelist = [col for col in original_X.columns if col in halogen_feature_names]
    
    whitelist_features = potential_whitelist
    print(f"Found {len(whitelist_features)} halogen features protected only in frequency/variance: {whitelist_features}")
    print("Using data from feature_data (Cell 0)")
    
elif 'optimized_results_fast' in globals():
    original_X = optimized_results_fast['original_features'] 
    y = optimized_results_fast['target_variable']
    whitelist_features = optimized_results_fast['whitelist_features']
    print("Using data from optimized_results_fast")
elif 'skip_mi_results_fast' in globals():
    original_X = skip_mi_results_fast['original_features'] 
    y = skip_mi_results_fast['target_variable']
    whitelist_features = skip_mi_results_fast['whitelist_features']
    print("Using data from skip_mi_results_fast")
else:
    print("Error: Please run Cell 0 (feature matrix generation) first")
    raise ValueError("Need to run Cell 0 to generate feature_data")


# UL-94 是二分类目标：V-0 vs 非 V-0。
y_encoded = normalize_ul94_target(y)
valid_target_mask = y_encoded.notna()
if not valid_target_mask.all():
    print(f"Dropping {int((~valid_target_mask).sum())} samples with unmapped UL-94 labels")
original_X = original_X.loc[valid_target_mask].reset_index(drop=True)
y = y_encoded.loc[valid_target_mask].astype(int).reset_index(drop=True)
print(f"UL-94 class distribution: {y.map(UL94_LABEL_NAMES).value_counts().to_dict()}")

print(f"Original features: {original_X.shape[1]}")
print(f"Samples: {len(y)}")
print(f"Halogen features protected in frequency/variance only: {len(whitelist_features)}")

# =============================================================================
# 完整参数网格定义
# =============================================================================

FREQ_THRESHOLDS = [0.01, 0.03, 0.05, 0.08, 0.1, 0.15]        # 6个
VAR_THRESHOLDS = [0.001, 0.005, 0.01, 0.02, 0.05]            # 5个  
MI_THRESHOLDS = [30, 40, 50, 60, 70]                          # 5个 (百分位)
CORR_THRESHOLDS = [0.8, 0.85, 0.9, 0.95, 0.98]               # 5个

TOTAL_COMBINATIONS = len(FREQ_THRESHOLDS) * len(VAR_THRESHOLDS) * len(MI_THRESHOLDS) * len(CORR_THRESHOLDS)

print(f"Parameter space: {len(FREQ_THRESHOLDS)}×{len(VAR_THRESHOLDS)}×{len(MI_THRESHOLDS)}×{len(CORR_THRESHOLDS)} = {TOTAL_COMBINATIONS} combinations")

# =============================================================================
# 核心支持函数
# =============================================================================

class StratifiedKFoldRegressor:
    def __init__(self, n_splits=5, shuffle=True, random_state=None, n_bins=5):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
        self.n_bins = n_bins
        
    def split(self, X, y, groups=None):
        stratified_bins, _ = create_stratified_bins(y, n_bins=self.n_bins)
        skf = StratifiedKFold(
            n_splits=self.n_splits, 
            shuffle=self.shuffle, 
            random_state=self.random_state
        )
        for train_idx, test_idx in skf.split(X, stratified_bins):
            yield train_idx, test_idx
    
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

def create_stratified_bins(y, n_bins=5):
    bin_edges = np.percentile(y, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    
    if len(bin_edges) <= n_bins:
        bin_edges = np.linspace(y.min(), y.max(), n_bins + 1)
    
    bins = pd.cut(y, bins=bin_edges, labels=False, include_lowest=True, duplicates='drop')
    bins = np.nan_to_num(bins, nan=0).astype(int)
    
    return bins, bin_edges

def safe_pearsonr(x, y):
    """安全计算皮尔逊相关系数"""
    try:
        mask = ~(np.isnan(x) | np.isnan(y))
        if mask.sum() < 2:
            return np.nan
        
        if np.var(x[mask]) == 0 or np.var(y[mask]) == 0:
            return np.nan
            
        corr, _ = pearsonr(x[mask], y[mask])
        return corr if not np.isnan(corr) else np.nan
    except:
        return np.nan

def evaluate_with_strong_model_improved(X, y, n_iter=30, cv=5, random_state=66):
    """UL-94 分类评估器：返回 ROC-AUC。"""
    if X.shape[1] == 0:
        return 0.0

    # SMILES 特征名可能含 [, ] 或 <，XGBoost 不接受这类 DataFrame 列名。
    X_model = X.to_numpy(dtype=np.float32, copy=False) if hasattr(X, 'to_numpy') else np.asarray(X, dtype=np.float32)

    try:
        y_arr = np.asarray(y).astype(int)
        classes, counts = np.unique(y_arr, return_counts=True)
        if not np.array_equal(classes, np.array([0, 1])):
            return 0.0
        n_splits = min(cv, int(counts.min()))
        if n_splits < 2:
            return 0.0

        param_distributions = {
            'n_estimators': randint(100, 500),
            'max_depth': randint(3, 8),
            'learning_rate': uniform(0.01, 0.19),
            'subsample': uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.6, 0.4),
            'min_child_weight': randint(1, 8),
            'gamma': uniform(0, 0.3),
            'reg_alpha': uniform(0, 0.5),
            'reg_lambda': uniform(0, 1.5)
        }

        base_model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            n_jobs=XGB_N_JOBS,
            tree_method='hist',
            random_state=random_state
        )
        scorer = SELECTION_SCORING
        cv_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_distributions,
            n_iter=n_iter,
            scoring=scorer,
            cv=cv_splitter,
            verbose=0,
            random_state=random_state,
            n_jobs=CV_SEARCH_N_JOBS
        )
        random_search.fit(X_model, y_arr)

        best_model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            n_jobs=XGB_N_JOBS,
            tree_method='hist',
            random_state=random_state,
            **random_search.best_params_
        )
        scores = cross_val_score(best_model, X_model, y_arr, cv=cv_splitter, scoring=scorer, n_jobs=CV_SEARCH_N_JOBS)
        return float(np.mean(scores))

    except Exception as e:
        print(f'Primary XGBoost evaluation failed; using fallback: {type(e).__name__}: {e}')
        try:
            y_arr = np.asarray(y).astype(int)
            classes, counts = np.unique(y_arr, return_counts=True)
            n_splits = min(cv, int(counts.min()))
            if not np.array_equal(classes, np.array([0, 1])) or n_splits < 2:
                return 0.0
            fallback_model = xgb.XGBClassifier(
                objective='binary:logistic',
                eval_metric='auc',
                n_estimators=200,
                max_depth=5,
                learning_rate=0.1,
                random_state=random_state,
                n_jobs=XGB_N_JOBS,
                tree_method='hist'
            )
            scorer = SELECTION_SCORING
            cv_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            scores = cross_val_score(fallback_model, X_model, y_arr, cv=cv_splitter, scoring=scorer, n_jobs=CV_SEARCH_N_JOBS)
            return float(np.mean(scores))
        except Exception as fallback_error:
            raise RuntimeError(f'XGBoost evaluation failed: {fallback_error}') from e

# =============================================================================
# 改进的相关性筛选函数
# =============================================================================

def improved_correlation_filtering(X_current, y, whitelist_features, corr_threshold):
    """
    改进的相关性筛选：
    1. 移除随机采样
    2. 修复早停机制
    3. 完整的成对比较
    """
    
    if len(X_current.columns) <= 1:
        return X_current.columns.tolist()
    
    try:
        # 完整相关矩阵仍保留，但用 NumPy 上三角一次找出候选对，避免数百万次 DataFrame.iloc。
        corr_values = X_current.corr().abs().to_numpy()
        target_correlations = X_current.corrwith(pd.Series(y, index=X_current.index)).abs().fillna(0).to_numpy()
        features = list(X_current.columns)
        pair_i, pair_j = np.where(np.triu(corr_values > corr_threshold, k=1))
        
        # np.where 以行优先顺序返回索引，和原 i/j 双循环的处理顺序一致。
        removed_features = set()
        whitelist_set = set(whitelist_features)
        for i, j in zip(pair_i, pair_j):
            feat1, feat2 = features[i], features[j]
            if feat1 in removed_features or feat2 in removed_features:
                continue
            
            if feat1 in whitelist_set and feat2 not in whitelist_set:
                removed_features.add(feat2)
            elif feat2 in whitelist_set and feat1 not in whitelist_set:
                removed_features.add(feat1)
            elif target_correlations[i] >= target_correlations[j]:
                removed_features.add(feat2)
            else:
                removed_features.add(feat1)
        
        # 5. 返回保留的特征
        final_features = [f for f in features if f not in removed_features]
        
        return final_features
        
    except Exception as e:
        # 相关性计算失败时，返回所有特征
        return X_current.columns.tolist()

# =============================================================================
# 改进的完整阈值流水线
# =============================================================================

def improved_complete_threshold_pipeline(X, y, whitelist_features, 
                                       freq_th, var_th, mi_th, corr_th):
    """
    改进的完整四步筛选流水线：
    1. 移除随机采样
    2. 修复早停机制  
    3. Br/F/Cl 只在频率和方差阶段保护；互信息、相关性和最终性能阶段不再保护
    4. 改进异常处理
    """
    
    X_current = X.copy()
    protected_features = [f for f in whitelist_features if f in X.columns]
    late_stage_protected_features: List[str] = []
    
    # 1. 频率筛选
    try:
        feature_prevalence = (X_current > 0).mean()
        freq_features = feature_prevalence[feature_prevalence >= freq_th].index.tolist()
        freq_features = ordered_union(freq_features, protected_features)
        
        if len(freq_features) == 0:
            return 0, 0.0
        
        X_current = X_current[freq_features]
        
    except Exception as e:
        return 0, 0.0
    
    # 2. 方差筛选
    try:
        var_selector = VarianceThreshold(threshold=var_th)
        var_selector.fit(X_current)
        var_features = X_current.columns[var_selector.get_support()].tolist()
        var_features = ordered_union(var_features, protected_features)
        
        if len(var_features) == 0:
            return 0, 0.0
        
        X_current = X_current[var_features]
        
    except Exception as e:
        # 方差筛选失败时，跳过此步骤
        pass
    
    # 3. 互信息筛选
    try:
        mi_selector = SelectKBest(MI_FUNC, k='all')
        mi_selector.fit(X_current, y)
        mi_scores = mi_selector.scores_
        
        valid_mask = ~np.isnan(mi_scores)
        if valid_mask.sum() > 0:
            mi_threshold = np.percentile(mi_scores[valid_mask], mi_th)
            mi_features = X_current.columns[(mi_scores > mi_threshold) & valid_mask].tolist()
        else:
            mi_features = X_current.columns.tolist()
            
        # 互信息阶段不再对白名单/卤素特征做保护。
        
        if len(mi_features) == 0:
            return 0, 0.0
        
        X_current = X_current[mi_features]
        
    except Exception as e:
        # 互信息筛选失败时，跳过此步骤
        pass
    
    # 4. 改进的相关性筛选（无采样）
    try:
        final_features = improved_correlation_filtering(
            X_current, y, late_stage_protected_features, corr_th
        )
        
        if len(final_features) == 0:
            return 0, 0.0
        
        X_current = X_current[final_features]
        
    except Exception as e:
        # 相关性筛选失败时，保持当前特征
        pass
    
    # 5. 不再进行性能动态白名单保护。
    
    final_feature_count = len(X_current.columns)
    
    if final_feature_count == 0:
        return 0, 0.0
    
    # 最终性能评估
    try:
        performance = evaluate_with_strong_model_improved(X_current, y)
        return final_feature_count, performance
    except Exception as e:
        return final_feature_count, 0.0

# =============================================================================
# 改进的750组合分析主函数
# =============================================================================

def improved_combined_threshold_sensitivity_analysis(X, y, whitelist_features, 
                                                   save_checkpoint=True):
    """改进的750组合四维阈值敏感性分析"""
    print("\nStarting improved 750-combination analysis...")
    print("=" * 70)
    
    # 生成所有750个组合
    all_combinations = list(product(FREQ_THRESHOLDS, VAR_THRESHOLDS, MI_THRESHOLDS, CORR_THRESHOLDS))
    
    # 检查是否有检查点文件
    checkpoint_file = os.path.join(plots_data_dir, 'improved_750_checkpoint.csv')
    
    if save_checkpoint and os.path.exists(checkpoint_file):
        print("Found checkpoint file, resuming from last position...")
        existing_results = pd.read_csv(checkpoint_file)
        completed_combinations = set()
        for _, row in existing_results.iterrows():
            combo = (row['freq_threshold'], row['var_threshold'], 
                    row['mi_threshold'], row['corr_threshold'])
            completed_combinations.add(combo)
        
        results = existing_results.to_dict('records')
        start_idx = len(results)
        print(f"Resuming from combination {start_idx}/{TOTAL_COMBINATIONS}")
    else:
        results = []
        completed_combinations = set()
        start_idx = 0
    
    start_time = time.time()
    
    # 进度条
    with tqdm(total=TOTAL_COMBINATIONS, initial=start_idx, 
              desc="Testing improved 750 combinations", unit="combo") as pbar:
        
        for combo_id, (freq_th, var_th, mi_th, corr_th) in enumerate(all_combinations):
            
            # 跳过已完成的组合
            if (freq_th, var_th, mi_th, corr_th) in completed_combinations:
                pbar.update(1)
                continue
            
            try:
                # 执行改进的四步筛选流水线
                feature_count, performance = improved_complete_threshold_pipeline(
                    X, y, whitelist_features, freq_th, var_th, mi_th, corr_th
                )
                
                results.append({
                    'combination_id': combo_id,
                    'freq_threshold': freq_th,
                    'var_threshold': var_th,
                    'mi_threshold': mi_th,
                    'corr_threshold': corr_th,
                    'feature_count': feature_count,
                    'performance': performance
                })
                
            except Exception as e:
                print(f"Combination {combo_id} failed: {e}")
                results.append({
                    'combination_id': combo_id,
                    'freq_threshold': freq_th,
                    'var_threshold': var_th,
                    'mi_threshold': mi_th,
                    'corr_threshold': corr_th,
                    'feature_count': 0,
                    'performance': 0.0
                })
            
            # 更新进度条
            current_best = max([r['performance'] for r in results]) if results else 0.0
            pbar.set_postfix({
                'Current ROC-AUC': f'{results[-1]["performance"]:.4f}' if results else '0.0000',
                'Best ROC-AUC': f'{current_best:.4f}',
                'Features': results[-1]["feature_count"] if results else 0
            })
            pbar.update(1)
            
            # 定期保存检查点
            if save_checkpoint and len(results) % 50 == 0:
                temp_df = pd.DataFrame(results)
                temp_df.to_csv(checkpoint_file, index=False)
                
                # 显示中间统计
                if len(results) % 100 == 0:
                    elapsed = (time.time() - start_time) / 60
                    remaining = (elapsed / len(results)) * (TOTAL_COMBINATIONS - len(results))
                    
                    best_combo = temp_df.loc[temp_df['performance'].idxmax()]
                    print(f"\n[{len(results)}/{TOTAL_COMBINATIONS}] Elapsed: {elapsed:.1f}min, Est. remaining: {remaining:.1f}min")
                    print(f"Current best: freq={best_combo['freq_threshold']}, var={best_combo['var_threshold']}, "
                          f"mi={best_combo['mi_threshold']}, corr={best_combo['corr_threshold']}")
                    print(f"Performance: ROC-AUC={best_combo['performance']:.4f}, Features={best_combo['feature_count']}")
    
    # 保存最终结果
    results_df = pd.DataFrame(results)
    final_file = os.path.join(plots_data_dir, 'improved_750_threshold_analysis.csv')
    results_df.to_csv(final_file, index=False)
    
    elapsed_time = (time.time() - start_time) / 60
    print(f"\nImproved 750-combination analysis completed in {elapsed_time:.1f} minutes!")
    print(f"Results saved to: {final_file}")
    
    return results_df

# =============================================================================
# 可视化和分析函数
# =============================================================================

def plot_improved_750_results(results_df):
    """绘制改进的750组合结果"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. 性能分布对比
    ax = axes[0, 0]
    ax.hist(results_df['performance'], bins=30, alpha=0.7, color='lightblue', edgecolor='black')
    mean_perf = results_df['performance'].mean()
    max_perf = results_df['performance'].max()
    ax.axvline(mean_perf, color='red', linestyle='--', label=f'Mean: {mean_perf:.4f}')
    ax.axvline(max_perf, color='orange', linestyle='--', label=f'Max: {max_perf:.4f}')
    ax.set_xlabel('ROC-AUC Score')
    ax.set_ylabel('Frequency')
    ax.set_title('Improved 750-Combination Performance Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. 特征数量 vs 性能
    ax = axes[0, 1]
    scatter = ax.scatter(results_df['feature_count'], results_df['performance'], 
                        c=results_df['performance'], cmap='RdYlBu_r', alpha=0.6, s=10)
    plt.colorbar(scatter, ax=ax, label='ROC-AUC Score')
    ax.set_xlabel('Feature Count')
    ax.set_ylabel('ROC-AUC Score')
    ax.set_title('Feature Count vs Performance (Improved)')
    ax.grid(True, alpha=0.3)
    
    # 3. 最优配置
    ax = axes[0, 2]
    best_config = results_df.loc[results_df['performance'].idxmax()]
    
    param_names = ['Frequency', 'Variance', 'MI Percentile', 'Correlation']
    param_values = [best_config['freq_threshold'], best_config['var_threshold'], 
                   best_config['mi_threshold'], best_config['corr_threshold']]
    
    bars = ax.bar(param_names, param_values, color=['blue', 'green', 'orange', 'red'], alpha=0.7)
    ax.set_ylabel('Parameter Value')
    ax.set_title(f'Optimal Configuration (Improved)\nROC-AUC={best_config["performance"]:.4f}, Features={best_config["feature_count"]}')
    ax.tick_params(axis='x', rotation=45)
    
    for bar, value in zip(bars, param_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(param_values) * 0.01,
               f'{value}', ha='center', va='bottom')
    
    # 4. 参数边际效应 - 频率
    ax = axes[1, 0]
    freq_effect = results_df.groupby('freq_threshold')['performance'].agg(['mean', 'std'])
    ax.errorbar(freq_effect.index, freq_effect['mean'], yerr=freq_effect['std'], 
               marker='o', capsize=5, capthick=2)
    ax.set_xlabel('Frequency Threshold')
    ax.set_ylabel('Mean ROC-AUC Score')
    ax.set_title('Frequency Marginal Effect (Improved)')
    ax.grid(True, alpha=0.3)
    
    # 5. 参数边际效应 - 方差
    ax = axes[1, 1]
    var_effect = results_df.groupby('var_threshold')['performance'].agg(['mean', 'std'])
    ax.errorbar(var_effect.index, var_effect['mean'], yerr=var_effect['std'], 
               marker='o', capsize=5, capthick=2)
    ax.set_xlabel('Variance Threshold')
    ax.set_ylabel('Mean ROC-AUC Score')
    ax.set_title('Variance Marginal Effect (Improved)')
    ax.grid(True, alpha=0.3)
    
    # 6. Top 10 配置对比
    ax = axes[1, 2]
    top_10 = results_df.nlargest(10, 'performance')
    
    y_pos = np.arange(len(top_10))
    bars = ax.barh(y_pos, top_10['performance'], alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f'#{i+1}' for i in range(len(top_10))])
    ax.set_xlabel('ROC-AUC Score')
    ax.set_title('Top 10 Configurations (Improved)')
    ax.grid(True, alpha=0.3)
    
    for i, (bar, perf) in enumerate(zip(bars, top_10['performance'])):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
               f'{perf:.4f}', ha='left', va='center', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_data_dir, 'improved_750_analysis_results.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    # 返回最优配置信息
    best_config = results_df.loc[results_df['performance'].idxmax()]
    return best_config

def save_plot_data(data, filename):
    """保存图表数据"""
    filepath = os.path.join(plots_data_dir, filename)
    if isinstance(data, pd.DataFrame):
        data.to_csv(filepath, index=False)
    print(f"Data saved: {filepath}")

# =============================================================================
# 主运行函数
# =============================================================================

def run_improved_750_analysis():
    """运行改进的750组合分析；审稿复现模式下优先读取已有750组合缓存。"""

    print("Starting improved 750-combination analysis with reproducible settings...")
    print("=" * 70)

    start_time = time.time()
    ran_750_search = False

    complete_results_file = os.path.join(plots_data_dir, 'improved_750_complete_results.csv')
    threshold_results_file = os.path.join(plots_data_dir, 'improved_750_threshold_analysis.csv')
    checkpoint_file = os.path.join(plots_data_dir, 'improved_750_checkpoint.csv')

    # 审稿复现模式：优先读取已经保存的750组合结果，避免重新搜索带来的结果漂移
    if USE_750_CACHE and (not RECOMPUTE_750_SEARCH) and os.path.exists(complete_results_file):
        print(f"Reproducibility mode: loading existing 750-combination results from {complete_results_file}")
        results_df = pd.read_csv(complete_results_file)
    elif USE_750_CACHE and (not RECOMPUTE_750_SEARCH) and os.path.exists(threshold_results_file):
        print(f"Reproducibility mode: loading existing threshold results from {threshold_results_file}")
        results_df = pd.read_csv(threshold_results_file)
    elif USE_750_CACHE and (not RECOMPUTE_750_SEARCH) and os.path.exists(checkpoint_file):
        print(f"Reproducibility mode: loading existing checkpoint from {checkpoint_file}")
        results_df = pd.read_csv(checkpoint_file)
    else:
        print("No reusable 750-combination cache found, or RECOMPUTE_750_SEARCH=True. Running full search...")
        ran_750_search = True
        results_df = improved_combined_threshold_sensitivity_analysis(
            original_X, y, whitelist_features,
            save_checkpoint=USE_750_CACHE
        )

    required_cols = {'freq_threshold', 'var_threshold', 'mi_threshold', 'corr_threshold',
                     'feature_count', 'performance'}
    missing_cols = required_cols - set(results_df.columns)
    if missing_cols:
        raise ValueError(f"750-combination results are missing required columns: {missing_cols}")

    # 生成可视化和分析；该步骤只基于已有结果表，不重新筛选
    print("\nGenerating analysis and visualizations from 750-combination table...")
    best_config = plot_improved_750_results(results_df)

    # 保存/刷新完整结果表
    save_plot_data(results_df, 'improved_750_complete_results.csv')

    elapsed_time = (time.time() - start_time) / 60

    print(f"\nImproved 750-Combination Analysis Results")
    print("=" * 60)
    print(f"Total analysis time: {elapsed_time:.1f} minutes")

    print(f"\nBEST CONFIGURATION FOUND:")
    print(f"  Frequency threshold: {best_config['freq_threshold']}")
    print(f"  Variance threshold: {best_config['var_threshold']}")
    print(f"  MI percentile: {best_config['mi_threshold']}%")
    print(f"  Correlation threshold: {best_config['corr_threshold']}")
    print(f"  Features: {best_config['feature_count']}")
    print(f"  ROC-AUC Score: {best_config['performance']:.4f}")

    print(f"\nReproducibility fixes applied:")
    print(f"  ✓ 750 cache reuse controlled by USE_750_CACHE / RECOMPUTE_750_SEARCH")
    print(f"  ✓ Ordered feature merging instead of set().union()")
    print(f"  ✓ Fixed random_state for mutual_info_classif")
    print(f"  ✓ XGBoost uses {XGB_N_JOBS} CPU threads; CV search uses {CV_SEARCH_N_JOBS} processes")
    print(f"  ✓ Final feature matrix can be anchored by REPRODUCE_PAPER_FEATURES")

    global improved_750_results
    improved_750_results = {
        'results_df': results_df,
        'best_config': best_config,
        'analysis_time': elapsed_time,
        'ran_750_search': ran_750_search
    }

    print(f"\nAll results saved in: {plots_data_dir}")
    print("=" * 60)

    return improved_750_results

# =============================================================================
# 最佳参数详细步骤分析
# =============================================================================

def detailed_stepwise_analysis_with_best_improved_params(X, y, whitelist_features, best_params):
    """
    使用最佳参数进行详细的逐步分析，记录每一步的R²
    """
    
    print("\n" + "="*80)
    print("DETAILED STEP-BY-STEP ANALYSIS WITH BEST IMPROVED PARAMETERS")
    print("="*80)
    
    print("Using best parameters from improved 750-combination analysis:")
    print(f"  Frequency threshold: {best_params['freq_threshold']}")
    print(f"  Variance threshold: {best_params['var_threshold']}")
    print(f"  MI percentile: {best_params['mi_threshold']}%")
    print(f"  Correlation threshold: {best_params['corr_threshold']}")
    print()
    
    start_time = time.time()
    step_results = []
    X_current = X.copy()
    protected_features = [f for f in whitelist_features if f in X.columns]
    late_stage_protected_features: List[str] = []
    
    # 初始状态：与后续阶段一样使用真实 ROC-AUC，避免 CSV 与控制台出现估算值。
    print(f"Initial state: {X_current.shape[1]} features")
    print("Evaluating original feature set so the CSV and console use the same real ROC-AUC values...")
    initial_r2 = evaluate_with_strong_model_improved(X_current, y)
    step_results.append({
        'step': 'Original',
        'feature_count': X_current.shape[1],
        'performance': initial_r2,
        'protected_count': 0,
        'removed_count': 0,
        'step_description': 'Initial feature set'
    })
    print(f"Initial ROC-AUC: {initial_r2:.4f}")
    print()
    
    # =========================================================================
    # 步骤1: 频率筛选
    # =========================================================================
    print("Step 1: Frequency filtering...")
    print(f"  Threshold: {best_params['freq_threshold']}")
    
    try:
        feature_prevalence = (X_current > 0).mean()
        freq_features = feature_prevalence[feature_prevalence >= best_params['freq_threshold']].index.tolist()
        
        # 白名单保护
        naturally_selected = freq_features.copy()
        freq_features = ordered_union(freq_features, protected_features)
        protected_in_freq = len([f for f in protected_features if f not in naturally_selected])
        
        features_before_freq = len(X_current.columns)
        X_current = X_current[freq_features]
        
        print(f"  Features after frequency filtering: {len(freq_features)}")
        print(f"  Protected features: {protected_in_freq}")
        
        # 评估性能
        freq_r2 = evaluate_with_strong_model_improved(X_current, y)
        step_results.append({
            'step': 'Frequency',
            'feature_count': len(freq_features),
            'performance': freq_r2,
            'protected_count': protected_in_freq,
            'removed_count': features_before_freq - len(freq_features),
            'step_description': f'Frequency >= {best_params["freq_threshold"]}'
        })
        print(f"  ROC-AUC after frequency filtering: {freq_r2:.4f}")
        
    except Exception as e:
        print(f"  Error in frequency filtering: {e}")
        freq_r2 = initial_r2
        step_results.append({
            'step': 'Frequency',
            'feature_count': len(X_current.columns),
            'performance': freq_r2,
            'protected_count': 0,
            'removed_count': 0,
            'step_description': 'Frequency filtering failed'
        })
    print()
    
    # =========================================================================
    # 步骤2: 方差筛选
    # =========================================================================
    print("Step 2: Variance filtering...")
    print(f"  Threshold: {best_params['var_threshold']}")
    
    try:
        var_selector = VarianceThreshold(threshold=best_params['var_threshold'])
        var_selector.fit(X_current)
        var_features = X_current.columns[var_selector.get_support()].tolist()
        
        # 白名单保护
        naturally_selected = var_features.copy()
        var_features = ordered_union(var_features, protected_features)
        protected_in_var = len([f for f in protected_features if f in X_current.columns and f not in naturally_selected])
        
        features_before_var = len(X_current.columns)
        X_current = X_current[var_features]
        
        print(f"  Features after variance filtering: {len(var_features)}")
        print(f"  Protected features: {protected_in_var}")
        
        # 评估性能
        var_r2 = evaluate_with_strong_model_improved(X_current, y)
        step_results.append({
            'step': 'Variance',
            'feature_count': len(var_features),
            'performance': var_r2,
            'protected_count': protected_in_var,
            'removed_count': features_before_var - len(var_features),
            'step_description': f'Variance >= {best_params["var_threshold"]}'
        })
        print(f"  ROC-AUC after variance filtering: {var_r2:.4f}")
        
    except Exception as e:
        print(f"  Error in variance filtering: {e}")
        var_r2 = freq_r2
        step_results.append({
            'step': 'Variance',
            'feature_count': len(X_current.columns),
            'performance': var_r2,
            'protected_count': 0,
            'removed_count': 0,
            'step_description': 'Variance filtering failed'
        })
    print()
    
    # =========================================================================
    # 步骤3: 互信息筛选
    # =========================================================================
    print("Step 3: Mutual information filtering...")
    print(f"  Percentile threshold: {best_params['mi_threshold']}%")
    
    try:
        mi_selector = SelectKBest(MI_FUNC, k='all')
        mi_selector.fit(X_current, y)
        mi_scores = mi_selector.scores_
        
        valid_mask = ~np.isnan(mi_scores)
        if valid_mask.sum() > 0:
            mi_threshold = np.percentile(mi_scores[valid_mask], best_params['mi_threshold'])
            mi_features = X_current.columns[(mi_scores > mi_threshold) & valid_mask].tolist()
        else:
            print("  Warning: All MI scores are NaN, keeping all features")
            mi_features = X_current.columns.tolist()
        
        # 互信息阶段不再保护 Br/F/Cl。
        protected_in_mi = 0
        
        features_before_mi = len(X_current.columns)
        X_current = X_current[mi_features]
        
        print(f"  Features after mutual information filtering: {len(mi_features)}")
        print(f"  Protected features: {protected_in_mi}")
        
        # 评估性能
        mi_r2 = evaluate_with_strong_model_improved(X_current, y)
        step_results.append({
            'step': 'MutualInfo',
            'feature_count': len(mi_features),
            'performance': mi_r2,
            'protected_count': protected_in_mi,
            'removed_count': features_before_mi - len(mi_features),
            'step_description': f'MI percentile >= {best_params["mi_threshold"]}%'
        })
        print(f"  ROC-AUC after mutual information filtering: {mi_r2:.4f}")
        
    except Exception as e:
        print(f"  Error in mutual information filtering: {e}")
        mi_r2 = var_r2
        step_results.append({
            'step': 'MutualInfo',
            'feature_count': len(X_current.columns),
            'performance': mi_r2,
            'protected_count': 0,
            'removed_count': 0,
            'step_description': 'MI filtering failed'
        })
    print()
    
    # =========================================================================
    # 步骤4: 改进的相关性筛选（无采样）
    # =========================================================================
    print("Step 4: Improved correlation filtering (no sampling)...")
    print(f"  Correlation threshold: {best_params['corr_threshold']}")
    
    try:
        features_before_corr = len(X_current.columns)
        print(f"  Processing {features_before_corr} features (no sampling)")
        
        # 使用改进的相关性筛选（无采样）
        final_features = improved_correlation_filtering(
            X_current, y, late_stage_protected_features, best_params['corr_threshold']
        )
        
        X_current = X_current[final_features]
        removed_in_corr = features_before_corr - len(final_features)
        
        print(f"  Removed {removed_in_corr} highly correlated features")
        print(f"  Features after correlation filtering: {len(final_features)}")
        
        # 评估性能
        corr_r2 = evaluate_with_strong_model_improved(X_current, y)
        step_results.append({
            'step': 'Correlation',
            'feature_count': len(final_features),
            'performance': corr_r2,
            'protected_count': 0,
            'removed_count': removed_in_corr,
            'step_description': f'Correlation < {best_params["corr_threshold"]} (improved)'
        })
        print(f"  ROC-AUC after correlation filtering: {corr_r2:.4f}")
        
    except Exception as e:
        print(f"  Error in correlation filtering: {e}")
        corr_r2 = mi_r2
        step_results.append({
            'step': 'Correlation',
            'feature_count': len(X_current.columns),
            'performance': corr_r2,
            'protected_count': 0,
            'removed_count': 0,
            'step_description': 'Correlation filtering failed'
        })
    print()
    
    # =========================================================================
    # 步骤5: 最终评估（不再做性能动态白名单保护）
    # =========================================================================
    print("Step 5: Final evaluation without performance-based whitelist protection...")
    final_r2 = corr_r2
    step_results.append({
        'step': 'Final',
        'feature_count': len(X_current.columns),
        'performance': final_r2,
        'protected_count': 0,
        'removed_count': 0,
        'step_description': 'No performance-based whitelist protection'
    })
    print(f"  Final ROC-AUC without dynamic whitelist protection: {final_r2:.4f}")
    
    # 分析完成
    elapsed_time = (time.time() - start_time) / 60
    
    print("\n" + "="*80)
    print("DETAILED STEP-BY-STEP ANALYSIS COMPLETE")
    print("="*80)
    print(f"Analysis time: {elapsed_time:.2f} minutes")
    print(f"Final feature count: {len(X_current.columns)}")
    print(f"Final ROC-AUC score: {final_r2:.4f}")
    
    # 步骤总结
    print(f"\nStep-by-step R² progression:")
    for result in step_results:
        step_name = result['step'].ljust(12)
        feature_count = str(result['feature_count']).rjust(6)
        performance = f"{result['performance']:.4f}"
        description = result['step_description']
        print(f"  {step_name}: {feature_count} features, ROC-AUC = {performance} ({description})")
    
    return step_results, X_current

def plot_detailed_stepwise_progression(step_results):
    """绘制详细的逐步分析结果"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 转换为DataFrame
    df = pd.DataFrame(step_results)
    
    # 1. 特征数量变化
    ax = axes[0, 0]
    bars = ax.bar(range(len(df)), df['feature_count'], alpha=0.7, color='steelblue')
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['step'], rotation=45)
    ax.set_ylabel('Feature Count')
    ax.set_title('Feature Count Changes (Best Improved Params)')
    
    for i, v in enumerate(df['feature_count']):
        ax.text(i, v + max(df['feature_count']) * 0.01, str(v), ha='center')
    
    # 2. 性能变化
    ax = axes[0, 1]
    line = ax.plot(range(len(df)), df['performance'], 'o-', linewidth=2, markersize=8, color='red')
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['step'], rotation=45)
    ax.set_ylabel('ROC-AUC Score')
    ax.set_title('Performance Changes (Best Improved Params)')
    ax.grid(True, alpha=0.3)
    
    for i, v in enumerate(df['performance']):
        marker = '*' if df.iloc[i]['step'] == 'Original' else ''
        ax.text(i, v + max(df['performance']) * 0.01, f'{v:.3f}{marker}', ha='center', va='bottom')
    
    # 3. 白名单保护效果
    ax = axes[1, 0]
    protected_data = df[df['protected_count'] > 0]
    if not protected_data.empty:
        bars = ax.bar(protected_data['step'], protected_data['protected_count'], 
                     alpha=0.7, color='orange')
        ax.set_ylabel('Protected Features Count')
        ax.set_title('Whitelist Protection Effect (Improved)')
        ax.tick_params(axis='x', rotation=45)
        
        for bar, count in zip(bars, protected_data['protected_count']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                   str(count), ha='center', va='bottom')
    else:
        ax.text(0.5, 0.5, 'No whitelist protection needed', ha='center', va='center', 
               transform=ax.transAxes)
        ax.set_title('Whitelist Protection Effect')
    
    # 4. 性能改进轨迹
    ax = axes[1, 1]
    if len(df) > 1:
        baseline = df.iloc[0]['performance']
        improvements = [row['performance'] - baseline for _, row in df.iterrows()]
        colors = ['green' if x >= 0 else 'red' for x in improvements]
        
        bars = ax.bar(range(len(df)), improvements, color=colors, alpha=0.7)
        ax.set_xticks(range(len(df)))
        ax.set_xticklabels(df['step'], rotation=45)
        ax.set_ylabel('R² Change from Original')
        ax.set_title('Performance Improvement Trajectory')
        ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        
        for bar, change in zip(bars, improvements):
            ax.text(bar.get_x() + bar.get_width()/2, 
                   bar.get_height() + (0.01 if change >= 0 else -0.01), 
                   f'{change:+.3f}', ha='center', va='bottom' if change >= 0 else 'top')
    
    plt.tight_layout()
    plt.savefig(os.path.join(plots_data_dir, 'best_improved_params_stepwise.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()

# =============================================================================
# 运行改进的750组合分析
# =============================================================================

print("\n🚀 READY TO RUN REPRODUCIBLE FEATURE-SELECTION ANALYSIS!")
print("Current reproducibility settings:")
print(f"• REPRODUCE_PAPER_FEATURES = {REPRODUCE_PAPER_FEATURES}")
print(f"• USE_750_CACHE = {USE_750_CACHE}")
print(f"• RECOMPUTE_750_SEARCH = {RECOMPUTE_750_SEARCH}")
print(f"• RECOMPUTE_FINAL_FEATURES = {RECOMPUTE_FINAL_FEATURES}")
print(f"• XGB_N_JOBS = {XGB_N_JOBS}; CV_SEARCH_N_JOBS = {CV_SEARCH_N_JOBS}")
print()
print("Default behavior:")
print("• Run or resume the 750-combination search when RECOMPUTE_750_SEARCH=True")
print("• Generate matching stepwise performance and final features after a fresh search")
print("• Reuse final features only when a 750 cache is deliberately loaded")
print("=" * 70)

# Override the legacy single-level scorer with the reproducible nested 5×3 protocol.
def evaluate_with_strong_model_improved(X, y, n_iter=N_ITER_SEARCH, outer_cv=OUTER_CV_SPLITS, inner_cv=INNER_CV_SPLITS, random_state=RANDOM_STATE):
    if X.shape[1] == 0:
        return 0.0
    from sklearn.model_selection import StratifiedKFold
    X_model = X.to_numpy(dtype=np.float32, copy=False) if hasattr(X, 'to_numpy') else np.asarray(X, dtype=np.float32)
    y_model = np.asarray(y).astype(int)
    classes, counts = np.unique(y_model, return_counts=True)
    if len(classes) < 2 or counts.min() < outer_cv:
        raise ValueError(f'Nested 5×3 CV requires at least {outer_cv} samples in every class; counts={dict(zip(classes, counts))}')
    param_distributions = {'n_estimators': [100, 300, 600], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [2, 3, 5, 7], 'subsample': [0.7, 0.9, 1.0], 'colsample_bytree': [0.7, 0.9, 1.0]}
    scorer = SELECTION_SCORING
    scores = []
    for fold, (train_idx, valid_idx) in enumerate(StratifiedKFold(n_splits=outer_cv, shuffle=True, random_state=random_state).split(X_model, y_model), 1):
        base_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc', tree_method='hist', n_jobs=XGB_N_JOBS, random_state=random_state, verbosity=0)
        try:
            search = RandomizedSearchCV(base_model, param_distributions, n_iter=n_iter, scoring=scorer, cv=StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=random_state), random_state=random_state, n_jobs=CV_SEARCH_N_JOBS, refit=True, error_score='raise', verbose=0)
            search.fit(X_model[train_idx], y_model[train_idx])
            model = search.best_estimator_
        except Exception as exc:
            print(f'Nested fold {fold}: search failed; using defaults. {type(exc).__name__}: {exc}', flush=True)
            model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='auc', tree_method='hist', n_estimators=200, max_depth=5, learning_rate=0.1, n_jobs=XGB_N_JOBS, random_state=random_state, verbosity=0)
            model.fit(X_model[train_idx], y_model[train_idx])
        scores.append(roc_auc_score(y_model[valid_idx], model.predict_proba(X_model[valid_idx])[:, 1]))
    return float(np.mean(scores))

# 运行改进的分析
improved_750_analysis_results = run_improved_750_analysis()

print("\n🎯 IMPROVED 750-COMBINATION ANALYSIS COMPLETE!")
print("=" * 50)
print("The 750-combination result table has been loaded or generated.")

# =============================================================================
# 运行最佳参数的详细步骤分析
# =============================================================================

print("\n" + "="*70)
print("RUNNING DETAILED ANALYSIS WITH BEST IMPROVED PARAMETERS")
print("="*70)

# 获取最佳参数
best_config = improved_750_analysis_results['best_config']

# 运行详细的逐步分析；审稿复现模式下最终特征矩阵优先作为锚点
stepwise_results_file = os.path.join(plots_data_dir, 'best_improved_params_stepwise_results.csv')
final_features_file = os.path.join(results_dir, 'best_improved_final_features_matrix.csv')

if improved_750_analysis_results['ran_750_search']:
    # A new 750 search must immediately materialize matching stepwise scores and features.
    print("\n750 search completed in this run; generating final features and stepwise performance values...")
    step_results, final_X = detailed_stepwise_analysis_with_best_improved_params(
        original_X, y, whitelist_features, best_config
    )
    plot_detailed_stepwise_progression(step_results)
    step_results_df = pd.DataFrame(step_results)
    step_results_df.to_csv(stepwise_results_file, index=False)
    final_X.to_csv(final_features_file, index=False)

elif REPRODUCE_PAPER_FEATURES and os.path.exists(final_features_file):
    print("\nReproducibility mode: using existing final feature matrix as the paper anchor.")
    print(f"Loaded final feature matrix: {final_features_file}")
    final_X = pd.read_csv(final_features_file)

    if os.path.exists(stepwise_results_file):
        print("Found existing step-by-step results, reusing them.")
        step_results_df = pd.read_csv(stepwise_results_file)
        step_results = step_results_df.to_dict('records')
    else:
        print("Step-by-step results not found. Creating a minimal anchored summary without re-screening features.")
        final_r2 = evaluate_with_strong_model_improved(final_X, y)
        step_results = [{
            'step': 'FinalCached',
            'feature_count': len(final_X.columns),
            'performance': final_r2,
            'protected_count': 0,
            'removed_count': np.nan,
            'step_description': 'Loaded from best_improved_final_features_matrix.csv'
        }]
        step_results_df = pd.DataFrame(step_results)
        step_results_df.to_csv(stepwise_results_file, index=False)

elif os.path.exists(stepwise_results_file) and os.path.exists(final_features_file):
    print("\nFound existing detailed step-by-step results and final matrix, reusing them...")
    step_results_df = pd.read_csv(stepwise_results_file)
    step_results = step_results_df.to_dict('records')
    final_X = pd.read_csv(final_features_file)

else:
    if not improved_750_analysis_results['ran_750_search']:
        raise FileNotFoundError("Cached 750 results do not have a matching final feature matrix; run a fresh 750 search.")
    if not RECOMPUTE_FINAL_FEATURES:
        raise FileNotFoundError(
            "Final feature matrix was not found. For paper reproduction, please restore "
            f"{final_features_file}. If you intentionally want to regenerate it, set "
            "RECOMPUTE_FINAL_FEATURES=True and REPRODUCE_PAPER_FEATURES=False."
        )

    print("\nRECOMPUTE_FINAL_FEATURES=True: regenerating final features with deterministic settings...")
    step_results, final_X = detailed_stepwise_analysis_with_best_improved_params(
        original_X, y, whitelist_features, best_config
    )

    print("\nGenerating detailed step-by-step visualizations...")
    plot_detailed_stepwise_progression(step_results)

    print("\nSaving detailed step-by-step results...")
    step_results_df = pd.DataFrame(step_results)
    step_results_df.to_csv(stepwise_results_file, index=False)

    # 只有显式允许重新生成时，才覆盖最终特征矩阵
    final_X.to_csv(final_features_file, index=False)

# 保存特征详细信息
feature_info = []
for feature in final_X.columns:
    prevalence = (final_X[feature] > 0).mean()
    variance = final_X[feature].var()
    correlation = abs(safe_pearsonr(final_X[feature], y))
    is_whitelist = feature in whitelist_features
    
    feature_info.append({
        'feature_name': feature,
        'prevalence': prevalence,
        'variance': variance,
        'correlation': correlation,
        'is_whitelist': is_whitelist,
        'importance_score': correlation * prevalence if not np.isnan(correlation) else 0
    })

feature_df = pd.DataFrame(feature_info)
feature_df.to_csv(os.path.join(results_dir, 'best_improved_final_features_info.csv'), index=False)

print(f"\nDetailed analysis results saved:")
print(f"  Step-by-step results: {stepwise_results_file}")
print(f"  Final feature matrix: {final_features_file}")
print(f"  Feature information: {os.path.join(results_dir, 'best_improved_final_features_info.csv')}")


# 保存带样本信息和目标列的最终建模表，方便后续多模型预测时直接读取。
final_features_with_target_file = os.path.join(results_dir, 'best_improved_final_features_with_target.csv')
if 'feature_data' in globals() and 'processed_data' in feature_data:
    processed_for_model = feature_data['processed_data'].reset_index(drop=True)
else:
    processed_data_candidates = sorted(Path(results_dir).glob('*_vocab_processed_data.csv'))
    if not processed_data_candidates:
        raise FileNotFoundError("Cannot find *_vocab_processed_data.csv for final feature/target alignment")
    processed_for_model = pd.read_csv(processed_data_candidates[0]).reset_index(drop=True)

final_X_for_model = final_X.reset_index(drop=True)
if len(processed_for_model) != len(final_X_for_model):
    raise ValueError(
        f"Final feature rows ({len(final_X_for_model)}) do not match processed data rows ({len(processed_for_model)})."
    )
meta_columns_for_model = [col for col in ['DOI', 'type', 'polymer_family', 'PolymerName', 'UL-94', 'P_content', 'Dripping'] if col in processed_for_model.columns]
final_features_with_target = pd.concat(
    [processed_for_model[meta_columns_for_model], final_X_for_model],
    axis=1,
)
final_features_with_target.to_csv(final_features_with_target_file, index=False)
print(f"  Final features with target: {final_features_with_target_file}")

print("\n" + "="*70)
print("🎯 COMPLETE IMPROVED ANALYSIS FINISHED!")
print("="*70)
print("✓ 750-combination analysis with improved algorithms")
print("✓ Loaded/generated 750-combination best configuration")
print("✓ Final feature matrix anchored for paper reproduction")
print("✓ Br/F/Cl protection applied only in frequency and variance filtering")
print("✓ All results saved and visualized")
print("="*70)
